# 🤖 다중 종목 LSTM 예측 모델

**전제:** `01_data_collection.ipynb` 실행 완료 후 실행

단일 종목이 아닌 **여러 종목의 데이터를 합쳐** LSTM을 학습시켜
더 일반화된 예측 모델을 만들고, 전 종목에 예측을 적용해 순위표를 생성한다.

| 항목 | 내용 |
|------|------|
| 입력 | 과거 60거래일의 종가 + 기술적 지표 (11개 특성) |
| 출력 | 5거래일 후 예상 등락률 |
| 학습 | 20개 대형주 데이터 → 전 종목 적용 |

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import warnings
from tqdm import tqdm

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.optimizers import Adam

from src.data_collector import get_stock_list, load_stock
from src.technical_indicators import add_all_indicators

matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False
warnings.filterwarnings('ignore')
print('준비 완료!')

준비 완료!


In [2]:
# ─────────────────────────────────────────
# 설정
# ─────────────────────────────────────────
import random

LOOKBACK    = 60    # 입력: 과거 60거래일
FORECAST    = 5     # 출력: 5거래일 후 등락률
TRAIN_RATIO = 0.8
BATCH_SIZE  = 256
EPOCHS      = 50

FEATURES = ['Close', 'Volume', 'RSI', 'MACD', 'MACD_Signal',
            'BB_Pct', 'BB_Width', 'MA_20', 'MA_60', 'ATR', 'Volume_Ratio']

# ── 층화 샘플링 설정 ──────────────────────────────────
# 대형주만 학습하면 소형주 예측 편향이 생긴다.
# 전체 시총 분포를 4구간으로 나누고 구간별로 동일한 수를
# 무작위 선택 → 모든 규모의 주가 패턴을 균등하게 학습.
STOCKS_PER_TIER = 50   # 구간당 종목 수
N_TIERS         = 4    # 구간 수 (대형 / 중대형 / 중소형 / 소형)
# 총 학습 종목 수 = STOCKS_PER_TIER × N_TIERS = 200개

# 하이퍼파라미터 탐색 셀 호환용 (20개 대형주 리스트 유지)
TRAIN_STOCKS = [
    '005930','000660','005380','000270','051910',
    '035420','035720','105560','055550','003670',
    '066570','028260','012330','096770','034730',
    '017670','030200','009150','018260','011200',
]

random.seed(42)
np.random.seed(42)

print(f'학습 전략: 시총 구간별 층화 샘플링')
print(f'  구간 수: {N_TIERS}개  ×  구간당 {STOCKS_PER_TIER}종목')
print(f'  총 학습 종목: {N_TIERS * STOCKS_PER_TIER}개')
print(f'입력 특성: {len(FEATURES)}개  |  시퀀스: {LOOKBACK}일 → {FORECAST}일 후')


In [3]:
# ─────────────────────────────────────────
# 1단계: 시총 구간별 층화 샘플링 + 데이터 준비
# ─────────────────────────────────────────

def prepare_stock_data(code, features, lookback, forecast):
    """단일 종목 데이터를 LSTM 입력 형태로 변환."""
    df = load_stock(code)
    if df is None or len(df) < lookback + forecast + 100:
        return None, None

    df = add_all_indicators(df)
    avail = [f for f in features if f in df.columns]
    df = df[avail].dropna()
    if len(df) < lookback + forecast + 50:
        return None, None

    # 종목별 독립 정규화 (종목 간 가격 스케일 차이 제거)
    scaler = MinMaxScaler()
    scaled = scaler.fit_transform(df)

    close_raw  = df['Close'].values
    future_ret = (close_raw[forecast:] - close_raw[:-forecast]) / close_raw[:-forecast]

    X, y = [], []
    for i in range(lookback, len(scaled) - forecast):
        X.append(scaled[i - lookback:i])
        y.append(future_ret[i - lookback])

    return np.array(X), np.array(y)


# ── Step 1: 층화 샘플링으로 학습 종목 선정 ──────────────
stock_list = get_stock_list()

# 시총 정보가 있고, 가격 파일도 존재하는 종목만 추림
price_dir    = '../data/prices'
available    = set(f.replace('.parquet','') for f in os.listdir(price_dir)
                   if f.endswith('.parquet'))
stock_list   = stock_list[
    stock_list['Code'].isin(available) &
    stock_list['MarketCap'].notna() &
    (stock_list['MarketCap'] > 0)
].copy()

# 시총 기준 4분위로 나누기
stock_list['Tier'] = pd.qcut(
    stock_list['MarketCap'],
    q=N_TIERS,
    labels=['소형주 (하위 25%)', '중소형주 (25~50%)',
            '중대형주 (50~75%)', '대형주 (상위 25%)']
)

# 구간별 현황 출력
print('=== 시총 구간별 종목 수 ===')
print(stock_list.groupby('Tier', observed=True)['Code'].count().to_string())

# 각 구간에서 STOCKS_PER_TIER개 무작위 선택
sampled = (
    stock_list
    .groupby('Tier', observed=True)
    .apply(lambda g: g.sample(min(STOCKS_PER_TIER, len(g)), random_state=111))
    .reset_index(drop=True)
)

name_map   = stock_list.set_index('Code')['Name'].to_dict()   if 'Name'   in stock_list.columns else {}
market_map = stock_list.set_index('Code')['Market'].to_dict() if 'Market' in stock_list.columns else {}

print(f'\n=== 층화 샘플링 결과 ({len(sampled)}종목) ===')
print(sampled.groupby('Tier', observed=True)['Code'].count().to_string())
print('\n구간별 대표 종목 (상위 3개):')
for tier, grp in sampled.groupby('Tier', observed=True):
    names = [name_map.get(c, c) for c in grp['Code'].head(3)]
    print(f'  {tier}: {", ".join(names)}')


# ── Step 2: 선정된 200개 종목 데이터 로드 ───────────────
all_X, all_y, success_codes = [], [], []

print(f'\n데이터 준비 중 ({len(sampled)}개 종목)...')
for _, row in tqdm(sampled.iterrows(), total=len(sampled), desc='데이터 준비'):
    code = row['Code']
    X, y = prepare_stock_data(code, FEATURES, LOOKBACK, FORECAST)
    if X is not None and len(X) > 50:
        all_X.append(X)
        all_y.append(y)
        success_codes.append(code)

X_all = np.vstack(all_X)
y_all = np.concatenate(all_y)

# 구간별 샘플 수 확인 (편향 여부 점검)
sampled_success = sampled[sampled['Code'].isin(success_codes)]
print(f'\n사용 종목: {len(success_codes)}개')
print('구간별 실제 사용 종목 수:')
print(sampled_success.groupby('Tier', observed=True)['Code'].count().to_string())
print(f'\n총 샘플: {len(X_all):,}개  |  입력 형태: {X_all.shape}')
print(f'타깃 평균: {y_all.mean()*100:.2f}%  표준편차: {y_all.std()*100:.2f}%')


In [4]:
# ─────────────────────────────────────────
# 2단계: 훈련/테스트 분리
# ─────────────────────────────────────────
# 4개 구간 종목이 섞인 채로 무작위 분리
# → 훈련/테스트 모두 전 구간 종목을 고르게 포함
idx = np.random.permutation(len(X_all))
X_all, y_all = X_all[idx], y_all[idx]

split = int(len(X_all) * TRAIN_RATIO)
X_train, X_test = X_all[:split], X_all[split:]
y_train, y_test = y_all[:split], y_all[split:]

print(f'훈련: {len(X_train):,}개  |  테스트: {len(X_test):,}개')
print(f'(대형주 ~ 소형주 4개 구간 종목이 균등 혼합됨)')


In [5]:
# ─────────────────────────────────────────
# 3단계: LSTM 모델 구축
# ─────────────────────────────────────────

# ══════════════════════════════════════════
# LSTM(Long Short-Term Memory)이란?
# ══════════════════════════════════════════
# 사람은 주식 차트를 볼 때 '어제 오르다가 오늘 갑자기 꺾였네'처럼
# 과거 흐름을 기억하면서 지금 상황을 판단한다.
# LSTM은 이 '기억하면서 판단하는' 능력을 갖춘 딥러닝 모델이다.
#
# 일반 신경망(Dense)은 입력을 독립적으로 처리해 시간 순서를 무시하지만,
# LSTM은 이전 시점의 정보를 다음 시점으로 넘기는 '메모리 셀'을 갖고 있다.
#
# ┌─────────────────────────────────────────────────────┐
# │  LSTM의 핵심 — 3개의 게이트(Gate)                    │
# │                                                     │
# │  • 망각 게이트(Forget Gate)                          │
# │    : 과거 기억 중 '잊어도 될 것'을 결정               │
# │    예) 3개월 전 거래량 급등은 지금과 무관 → 잊기       │
# │                                                     │
# │  • 입력 게이트(Input Gate)                           │
# │    : 현재 입력 중 '기억할 것'을 결정                   │
# │    예) 오늘 RSI가 30 이하로 떨어짐 → 중요, 기억        │
# │                                                     │
# │  • 출력 게이트(Output Gate)                          │
# │    : 지금 메모리에서 '출력할 것'을 결정                │
# │    예) 현재 기억을 바탕으로 5일 후 등락률 예측 출력     │
# └─────────────────────────────────────────────────────┘
#
# 주가 예측에 LSTM을 쓰는 이유:
# - 주가는 '시간 순서'가 있는 데이터(시계열)다
# - 60일치 지표를 순서대로 보여주면 단기→중기 패턴을 계층적으로 학습
# - RF·XGBoost는 오늘 하루 데이터만 보지만,
#   LSTM은 60일의 흐름 전체를 보고 판단한다

n_features = X_train.shape[2]  # 11개 기술적 지표

model = Sequential([
    # ── 1층 LSTM: 단기 패턴 학습 ──────────────────────
    # input_shape=(60, 11): 60일치 × 11개 지표를 한 번에 입력
    # 128개 유닛: 각 유닛이 서로 다른 패턴(RSI 추세, 거래량 급등 등)을 담당
    # return_sequences=True: 각 날짜의 출력을 다음 층으로 전달 (시퀀스 유지)
    LSTM(128, return_sequences=True, input_shape=(LOOKBACK, n_features)),

    # BatchNormalization: 층 간 데이터 분포를 정규화해 학습을 안정시킴
    # → 없으면 층이 깊어질수록 값이 폭발하거나 소멸하는 문제 발생
    BatchNormalization(),

    # Dropout(0.2): 학습 중 랜덤으로 20% 유닛을 꺼서 과적합 방지
    # → 특정 패턴만 외우지 않고 일반적인 규칙을 학습하게 강제
    Dropout(0.2),

    # ── 2층 LSTM: 중기 패턴 학습 ──────────────────────
    # 64개 유닛: 1층의 단기 패턴을 조합해 더 추상적인 중기 패턴 추출
    # 예) '3일 연속 RSI 하락 + 거래량 증가' 같은 복합 패턴
    LSTM(64, return_sequences=True),
    BatchNormalization(),
    Dropout(0.2),

    # ── 3층 LSTM: 최종 시퀀스 압축 ────────────────────
    # 32개 유닛: 60일치 흐름을 하나의 벡터로 압축
    # return_sequences=False: 마지막 시점의 출력만 다음 층으로 전달
    # → '60일치를 요약한 하나의 표현'을 만드는 단계
    LSTM(32, return_sequences=False),
    Dropout(0.1),

    # ── 출력층: 등락률 예측 ────────────────────────────
    # Dense(16): LSTM 출력을 받아 비선형 변환 (ReLU 활성화)
    # → LSTM이 추출한 패턴을 최종 예측값으로 변환하는 과정
    Dense(16, activation='relu'),

    # Dense(1): 최종 출력 — 5거래일 후 등락률 1개 값
    # 활성화 함수 없음(선형): 등락률은 -∞~+∞ 범위의 연속값이므로
    Dense(1)
])

# Huber Loss:
# - 일반 MSE는 이상치(갑작스러운 급등락)에 민감해 학습이 흔들림
# - Huber는 오차가 작을 때는 MSE처럼, 클 때는 MAE처럼 동작
# → 주가의 갑작스러운 급등락에 강건한 학습 가능
model.compile(optimizer=Adam(learning_rate=0.001), loss='huber', metrics=['mae'])
model.summary()


In [ ]:
# ─────────────────────────────────────────
# 4단계: 모델 학습
# ─────────────────────────────────────────
os.makedirs('../models', exist_ok=True)

callbacks = [
    # EarlyStopping: 검증 손실이 7에포크 동안 개선되지 않으면 자동 종료
    # → 필요 이상으로 학습해서 과적합되는 것을 방지
    # restore_best_weights=True: 종료 시 가장 성능 좋았던 시점의 가중치 복원
    EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True),

    # ModelCheckpoint: 검증 손실이 개선될 때마다 모델 파일 저장
    # → 학습 중간에 프로그램이 꺼져도 최고 성능 모델을 보존
    ModelCheckpoint('../models/lstm_multistock.keras',
                    monitor='val_loss', save_best_only=True, verbose=0)
]

history = model.fit(
    X_train, y_train,
    epochs=EPOCHS,          # 최대 50번 반복 (EarlyStopping이 더 일찍 멈출 수 있음)
    batch_size=BATCH_SIZE,  # 한 번에 256개 샘플로 가중치 업데이트
                            # → 너무 작으면 학습 느리고 불안정, 너무 크면 일반화 저하
    validation_split=0.1,   # 훈련 데이터의 10%를 검증용으로 분리
                            # → 학습 중 과적합 여부를 실시간으로 모니터링
    callbacks=callbacks,
    verbose=1
)
print('\n학습 완료!')


In [ ]:
# 학습 곡선
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(history.history['loss'], label='훈련')
axes[0].plot(history.history['val_loss'], label='검증')
axes[0].set_title('손실 (Huber Loss)')
axes[0].set_xlabel('에포크')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['mae'], label='훈련')
axes[1].plot(history.history['val_mae'], label='검증')
axes[1].set_title('평균 절대 오차 (MAE)')
axes[1].set_xlabel('에포크')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('다중 종목 LSTM 학습 곡선', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────
# 5단계: 성능 평가
# ─────────────────────────────────────────
y_pred = model.predict(X_test, verbose=0).flatten()

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
dir_acc = np.mean(np.sign(y_pred) == np.sign(y_test))

print('=== 테스트 성능 ===')
print(f'MAE:          {mae*100:.2f}%')
print(f'RMSE:         {rmse*100:.2f}%')
print(f'방향 정확도:  {dir_acc*100:.1f}%  (50% 초과 = 무작위보다 낫다)')

plt.figure(figsize=(7, 6))
lim = max(abs(y_test).max(), abs(y_pred).max()) * 100 * 1.1
plt.scatter(y_test*100, y_pred*100, alpha=0.1, s=5, color='steelblue')
plt.plot([-lim, lim], [-lim, lim], 'r--', linewidth=1.5, label='완벽한 예측')
plt.xlim(-lim, lim)
plt.ylim(-lim, lim)
plt.xlabel('실제 등락률 (%)')
plt.ylabel('예측 등락률 (%)')
plt.title(f'실제 vs 예측  |  방향 정확도: {dir_acc*100:.1f}%')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────
# 6단계: 전 종목 예측 순위표 생성
# ─────────────────────────────────────────
price_dir  = '../data/prices'
files      = [f for f in os.listdir(price_dir) if f.endswith('.parquet')]
stock_list = get_stock_list()
name_map   = stock_list.set_index('Code')['Name'].to_dict()   if 'Name'   in stock_list.columns else {}
market_map = stock_list.set_index('Code')['Market'].to_dict() if 'Market' in stock_list.columns else {}

ranking = []
print(f'전 종목 예측 중: {len(files)}개...')
for fname in tqdm(files, desc='예측'):
    code = fname.replace('.parquet', '')
    X, _ = prepare_stock_data(code, FEATURES, LOOKBACK, FORECAST)
    if X is None or len(X) < 1:
        continue
    pred = float(model.predict(X[[-1]], verbose=0)[0][0])
    df_tmp = load_stock(code)
    cur_price = int(df_tmp['Close'].iloc[-1]) if df_tmp is not None else None
    ranking.append({
        '종목코드': code,
        '종목명':   name_map.get(code, code),
        '시장':     market_map.get(code, ''),
        '현재가':   cur_price,
        f'예측_{FORECAST}일후_등락률(%)': round(pred * 100, 2),
    })

rank_df = pd.DataFrame(ranking).sort_values(f'예측_{FORECAST}일후_등락률(%)', ascending=False)

print(f'\n=== LSTM 예측 상위 20종목 ({FORECAST}일 후 기준) ===')
print(rank_df.head(20).to_string(index=False))

os.makedirs('../data/stock_list', exist_ok=True)
rank_df.to_csv('../data/stock_list/lstm_ranking.csv', index=False, encoding='utf-8-sig')
print('\n순위표 저장 완료! → data/stock_list/lstm_ranking.csv')
print('다음: 05_backtesting.ipynb')

---
## 🔍 하이퍼파라미터 탐색

최종 모델을 확정하기 전에 각 모델의 핵심 파라미터를 체계적으로 비교했습니다.
단순히 기본값을 쓰는 것이 아니라 **실험을 통해 최적값을 결정**한 근거를 남깁니다.

### LSTM — 은닉층 유닛 수 탐색

| 파라미터 | 후보값 | 선택 기준 |
|----------|--------|-----------|
| 1층 유닛 수 | 32 / 64 / **128** / 256 | 검증 손실 최소 |
| 학습률 | 0.01 / **0.001** / 0.0003 | 수렴 속도 + 안정성 |

> 탐색 시간 절감을 위해 최대 **15에포크**만 학습하고 Early Stopping으로 조기 종료합니다.


In [ ]:
# ─────────────────────────────────────────
# [탐색] LSTM 은닉 유닛 수 비교
# 빠른 실험을 위해 최대 15에포크로 제한
# ─────────────────────────────────────────
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
import time

UNIT_CANDIDATES = [32, 64, 128, 256]
lstm_search_results = []

for units in UNIT_CANDIDATES:
    print(f'\n--- LSTM 유닛={units} 학습 중 ---')
    t0 = time.time()

    # 유닛 수에 비례한 3층 구조
    m = Sequential([
        LSTM(units,      return_sequences=True,  input_shape=(LOOKBACK, len(FEATURES))),
        BatchNormalization(), Dropout(0.2),
        LSTM(units // 2, return_sequences=True),
        BatchNormalization(), Dropout(0.2),
        LSTM(units // 4, return_sequences=False),
        Dropout(0.1),
        Dense(16, activation='relu'),
        Dense(1)
    ])
    m.compile(optimizer=Adam(0.001), loss='huber', metrics=['mae'])

    cb = EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True)
    hist = m.fit(
        X_train, y_train,
        epochs=15,           # 탐색이므로 15에포크로 제한
        batch_size=256,
        validation_split=0.1,
        callbacks=[cb],
        verbose=0
    )
    val_loss = min(hist.history['val_loss'])
    val_mae  = min(hist.history['val_mae'])
    elapsed  = time.time() - t0
    params   = m.count_params()

    lstm_search_results.append({
        '유닛수': units,
        '파라미터수': f'{params:,}',
        '최소 val_loss': round(val_loss, 6),
        '최소 val_MAE':  round(val_mae,  4),
        '학습시간(초)':  round(elapsed,   1),
        '_hist': hist.history   # 시각화용, 표에는 미표시
    })
    print(f'  val_loss={val_loss:.5f}  val_MAE={val_mae:.4f}  ({elapsed:.0f}초)')

# 결과 표 출력
lstm_sr_df = pd.DataFrame(lstm_search_results).drop(columns=['_hist'])
print('\n=== LSTM 유닛 수별 성능 비교 ===')
print(lstm_sr_df.to_string(index=False))

best_units = lstm_search_results[
    min(range(len(lstm_search_results)),
        key=lambda i: lstm_search_results[i]['최소 val_loss'])
]['유닛수']
print(f'\n★ 최적 유닛 수: {best_units}  → 최종 모델에 사용')

# ── 시각화: 유닛별 학습 곡선 ──
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
colors = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3']

for res, color in zip(lstm_search_results, colors):
    units = res['유닛수']
    axes[0].plot(res['_hist']['val_loss'], label=f'유닛={units}', color=color)
    axes[1].plot(res['_hist']['val_mae'],  label=f'유닛={units}', color=color)

axes[0].set_title('유닛 수별 검증 손실 (val_loss)')
axes[0].set_xlabel('에포크')
axes[0].set_ylabel('Huber Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_title('유닛 수별 검증 MAE (val_MAE)')
axes[1].set_xlabel('에포크')
axes[1].set_ylabel('MAE')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('LSTM 은닉 유닛 수 탐색 결과', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


### Random Forest — 트리 수 탐색

| 파라미터 | 후보값 | 선택 기준 |
|----------|--------|-----------|
| n_estimators (트리 수) | 50 / 100 / 200 / **300** / 500 | OOB 오차 + 테스트 MAE |

> 트리 수가 많을수록 성능이 안정되지만 학습 시간이 선형으로 증가합니다.  
> **성능 향상이 포화되는 지점(elbow point)**을 찾아 최적값을 결정합니다.


In [ ]:
# ─────────────────────────────────────────
# [탐색] Random Forest 트리 수(n_estimators) 비교
# ─────────────────────────────────────────
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import time

X_flat_train = X_train[:, -1, :]   # 마지막 타임스텝만 사용 (2D 플랫)
X_flat_test  = X_test[:,  -1, :]

N_ESTIMATORS_LIST = [50, 100, 200, 300, 500]
rf_search_results = []

for n in N_ESTIMATORS_LIST:
    t0 = time.time()
    m = RandomForestRegressor(
        n_estimators=n, max_depth=8, min_samples_leaf=20,
        n_jobs=-1, random_state=111,
        oob_score=True   # Out-of-Bag 오차로 내부 검증
    )
    m.fit(X_flat_train, y_train)
    pred      = m.predict(X_flat_test)
    test_mae  = mean_absolute_error(y_test, pred) * 100
    test_rmse = np.sqrt(mean_squared_error(y_test, pred)) * 100
    dir_acc   = np.mean(np.sign(pred) == np.sign(y_test)) * 100
    elapsed   = time.time() - t0

    rf_search_results.append({
        '트리 수': n,
        'OOB R²':     round(m.oob_score_, 4),
        'MAE(%)':     round(test_mae,  3),
        'RMSE(%)':    round(test_rmse, 3),
        '방향정확도': round(dir_acc,   1),
        '학습시간(초)': round(elapsed, 1),
        '_pred': pred
    })
    print(f'n={n:>3d}: MAE={test_mae:.3f}%  방향={dir_acc:.1f}%  ({elapsed:.1f}초)')

rf_sr_df = pd.DataFrame(rf_search_results).drop(columns=['_pred'])
print('\n=== RF 트리 수별 성능 비교 ===')
print(rf_sr_df.to_string(index=False))

best_n_rf = rf_search_results[
    min(range(len(rf_search_results)),
        key=lambda i: rf_search_results[i]['MAE(%)'])
]['트리 수']
print(f'\n★ 최적 트리 수: {best_n_rf}  → 최종 모델에 사용')

# ── 시각화 ──
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

ns = [r['트리 수'] for r in rf_search_results]

# MAE 추이
axes[0].plot(ns, [r['MAE(%)'] for r in rf_search_results],
             marker='o', color='seagreen', linewidth=2)
axes[0].axvline(best_n_rf, color='red', linestyle='--', alpha=0.7, label=f'최적={best_n_rf}')
axes[0].set_title('트리 수 vs MAE  (낮을수록 좋음)')
axes[0].set_xlabel('n_estimators')
axes[0].set_ylabel('MAE (%)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 방향정확도 추이
axes[1].plot(ns, [r['방향정확도'] for r in rf_search_results],
             marker='s', color='steelblue', linewidth=2)
axes[1].axhline(50, color='gray', linestyle='--', alpha=0.6, label='기준(50%)')
axes[1].axvline(best_n_rf, color='red', linestyle='--', alpha=0.7)
axes[1].set_title('트리 수 vs 방향정확도  (높을수록 좋음)')
axes[1].set_xlabel('n_estimators')
axes[1].set_ylabel('방향정확도 (%)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# 학습 시간 추이
axes[2].bar(ns, [r['학습시간(초)'] for r in rf_search_results],
            color='salmon', alpha=0.8, width=35)
axes[2].set_title('트리 수 vs 학습 시간')
axes[2].set_xlabel('n_estimators')
axes[2].set_ylabel('학습 시간 (초)')
axes[2].grid(True, alpha=0.3, axis='y')

plt.suptitle('Random Forest 트리 수(n_estimators) 탐색', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


### XGBoost — 부스팅 라운드 수 탐색

| 파라미터 | 후보값 | 선택 기준 |
|----------|--------|-----------|
| n_estimators (라운드 수) | 100 / 200 / 300 / **500** / 700 | Early Stopping 기반 최적 라운드 |

> XGBoost는 **Early Stopping**을 지원합니다.  
> 검증 손실이 개선되지 않으면 자동으로 학습을 멈춰 과적합을 방지합니다.  
> 각 후보별로 실제 종료 라운드를 기록해 과적합 경향을 확인합니다.


In [ ]:
# ─────────────────────────────────────────
# [탐색] XGBoost 부스팅 라운드 수 비교
# Early Stopping(patience=30)으로 최적 라운드 자동 탐색
# ─────────────────────────────────────────
from xgboost import XGBRegressor
import time

N_ROUNDS_LIST = [100, 200, 300, 500, 700]
xgb_search_results = []

for n in N_ROUNDS_LIST:
    t0 = time.time()
    m = XGBRegressor(
        n_estimators=n, max_depth=5, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        n_jobs=-1, random_state=111, verbosity=0,
        early_stopping_rounds=30   # 30라운드 개선 없으면 조기 종료
    )
    m.fit(
        X_flat_train, y_train,
        eval_set=[(X_flat_test, y_test)],
        verbose=False
    )
    best_round = m.best_iteration + 1   # 실제 사용된 라운드
    pred       = m.predict(X_flat_test)
    test_mae   = mean_absolute_error(y_test, pred) * 100
    test_rmse  = np.sqrt(mean_squared_error(y_test, pred)) * 100
    dir_acc    = np.mean(np.sign(pred) == np.sign(y_test)) * 100
    elapsed    = time.time() - t0

    xgb_search_results.append({
        '설정 라운드': n,
        '실제 사용': best_round,
        '조기종료 여부': '✅' if best_round < n else '❌',
        'MAE(%)':     round(test_mae,  3),
        'RMSE(%)':    round(test_rmse, 3),
        '방향정확도': round(dir_acc,   1),
        '학습시간(초)': round(elapsed, 1),
        '_pred': pred,
        '_best': best_round
    })
    early = '(조기종료)' if best_round < n else ''
    print(f'n={n:>3d} → 실사용={best_round:>3d}라운드 {early}  '
          f'MAE={test_mae:.3f}%  방향={dir_acc:.1f}%  ({elapsed:.1f}초)')

xgb_sr_df = pd.DataFrame(xgb_search_results).drop(columns=['_pred','_best'])
print('\n=== XGBoost 라운드 수별 성능 비교 ===')
print(xgb_sr_df.to_string(index=False))

best_n_xgb = xgb_search_results[
    min(range(len(xgb_search_results)),
        key=lambda i: xgb_search_results[i]['MAE(%)'])
]['설정 라운드']
print(f'\n★ 최적 라운드 수: {best_n_xgb}  → 최종 모델에 사용')

# ── 시각화 ──
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

ns_set  = [r['설정 라운드'] for r in xgb_search_results]
ns_best = [r['실제 사용']   for r in xgb_search_results]

# MAE 추이
axes[0].plot(ns_set, [r['MAE(%)'] for r in xgb_search_results],
             marker='o', color='tomato', linewidth=2)
axes[0].axvline(best_n_xgb, color='red', linestyle='--', alpha=0.7, label=f'최적={best_n_xgb}')
axes[0].set_title('라운드 수 vs MAE')
axes[0].set_xlabel('n_estimators (설정값)')
axes[0].set_ylabel('MAE (%)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 설정 vs 실제 사용 라운드 비교
x_pos = np.arange(len(ns_set))
w = 0.35
axes[1].bar(x_pos - w/2, ns_set,  w, label='설정 라운드', color='lightcoral', alpha=0.85)
axes[1].bar(x_pos + w/2, ns_best, w, label='실제 사용',   color='tomato',     alpha=0.85)
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(ns_set)
axes[1].set_title('설정 라운드 vs 실제 사용 라운드\n(Early Stopping 효과)')
axes[1].set_xlabel('설정 라운드')
axes[1].set_ylabel('라운드 수')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

# 방향정확도 추이
axes[2].plot(ns_set, [r['방향정확도'] for r in xgb_search_results],
             marker='s', color='steelblue', linewidth=2)
axes[2].axhline(50, color='gray', linestyle='--', alpha=0.6, label='기준(50%)')
axes[2].set_title('라운드 수 vs 방향정확도')
axes[2].set_xlabel('n_estimators (설정값)')
axes[2].set_ylabel('방향정확도 (%)')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.suptitle('XGBoost 부스팅 라운드 수 탐색', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


### 선형 계열 모델 — 규제 강도(Alpha) 탐색

선형 회귀에 과적합 방지 장치를 추가한 모델들입니다.

| 모델 | 설명 | Alpha 역할 |
|------|------|-----------|
| **선형 회귀** | 기준선. 규제 없음 | — |
| **Ridge** | 모든 계수를 조금씩 줄임 | 클수록 계수가 더 작아짐 |
| **Lasso** | 중요하지 않은 계수를 0으로 만듦 | 클수록 계수가 0이 되는 것이 많아짐 |
| **ElasticNet** | Ridge + Lasso 혼합 | 두 방식을 동시에 적용 |

> Alpha(α)가 너무 작으면 → 과적합 위험  
> Alpha(α)가 너무 크면 → 모든 계수가 0에 수렴 → 예측력 상실

In [ ]:
# ─────────────────────────────────────────
# [탐색] Ridge / Lasso / ElasticNet — 규제 강도(Alpha) 비교
# ─────────────────────────────────────────
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error
import time

# ══════════════════════════════════════════
# 선형 계열 모델이란?
# ══════════════════════════════════════════
# 선형 회귀는 '입력 × 가중치의 합 = 예측값' 방식이다.
# 가중치(계수)가 너무 크면 훈련 데이터만 외우는 과적합이 생기는데,
# 이를 막기 위해 '규제(Regularization)'를 추가한 것이 아래 3모델이다.
#
# Ridge:      모든 계수를 조금씩 줄임         (완전히 0이 되지는 않음)
# Lasso:      중요하지 않은 계수를 0으로 만듦  (자동 변수 선택 효과)
# ElasticNet: Ridge + Lasso 혼합              (두 장점 결합)
#
# Alpha(α): 규제 강도. 클수록 계수가 더 강하게 억제됨.

ALPHA_LIST = [0.0001, 0.001, 0.01, 0.1, 1.0, 10.0]
linear_search_results = []

print('=== Ridge / Lasso / ElasticNet Alpha 탐색 ===\n')
for alpha in ALPHA_LIST:
    row = {'alpha': alpha}
    for ModelClass, name in [(Ridge, 'Ridge'), (Lasso, 'Lasso'), (ElasticNet, 'ElasticNet')]:
        m = ModelClass(alpha=alpha, max_iter=2000)
        m.fit(X_flat_train, y_train)
        pred    = m.predict(X_flat_test)
        mae     = mean_absolute_error(y_test, pred) * 100
        dir_acc = np.mean(np.sign(pred) == np.sign(y_test)) * 100
        row[f'{name}_MAE']     = round(mae,     3)
        row[f'{name}_DirAcc']  = round(dir_acc, 1)
    linear_search_results.append(row)
    print(f'α={alpha:<8}  Ridge MAE={row["Ridge_MAE"]:.3f}%  '
          f'Lasso MAE={row["Lasso_MAE"]:.3f}%  '
          f'ElasticNet MAE={row["ElasticNet_MAE"]:.3f}%')

lr_df = pd.DataFrame(linear_search_results)
print('\n=== Alpha별 성능 상세 ===')
print(lr_df.to_string(index=False))

# 각 모델별 최적 alpha
for name in ['Ridge', 'Lasso', 'ElasticNet']:
    best_alpha = lr_df.loc[lr_df[f'{name}_MAE'].idxmin(), 'alpha']
    print(f'★ {name} 최적 alpha: {best_alpha}')

# ── 시각화 ──
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, name, color in zip(axes,
                            ['Ridge', 'Lasso', 'ElasticNet'],
                            ['royalblue', 'darkorange', 'purple']):
    ax.semilogx(lr_df['alpha'], lr_df[f'{name}_MAE'],
                marker='o', color=color, linewidth=2)
    best_a = lr_df.loc[lr_df[f'{name}_MAE'].idxmin(), 'alpha']
    ax.axvline(best_a, color='red', linestyle='--', alpha=0.7, label=f'최적 α={best_a}')
    ax.set_title(f'{name} — Alpha vs MAE')
    ax.set_xlabel('Alpha (로그 스케일)')
    ax.set_ylabel('MAE (%)')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('선형 계열 모델 규제 강도(Alpha) 탐색', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 결정 트리(Decision Tree) — 최대 깊이(max_depth) 탐색

| 파라미터 | 후보값 | 선택 기준 |
|----------|--------|-----------|
| max_depth (트리 깊이) | 2 / 4 / 6 / **8** / 12 / None | 테스트 MAE 최소 |

> 깊이가 얕으면 → 너무 단순해서 예측력 부족 (과소적합)  
> 깊이가 깊으면 → 훈련 데이터만 외움 (과적합)  
> **적절한 깊이**를 탐색해 두 극단 사이의 최적점을 찾습니다.

In [ ]:
# ─────────────────────────────────────────
# [탐색] 결정 트리 — 최대 깊이(max_depth) 비교
# ─────────────────────────────────────────
from sklearn.tree import DecisionTreeRegressor

# ══════════════════════════════════════════
# 결정 트리란?
# ══════════════════════════════════════════
# "RSI가 70 이상이면서 거래량이 평균보다 높으면 → 하락 예측"처럼
# 조건 분기를 반복해서 예측값에 도달하는 모델이다.
#
# 장점: 사람이 이해하기 쉬운 규칙 구조
# 단점: 깊어질수록 훈련 데이터의 노이즈까지 외움 (과적합)
# → max_depth로 깊이를 제한해 과적합을 막는다

DEPTH_LIST = [2, 4, 6, 8, 12, None]   # None = 제한 없음
dt_search_results = []

print('=== 결정 트리 최대 깊이 탐색 ===\n')
for depth in DEPTH_LIST:
    m = DecisionTreeRegressor(max_depth=depth, min_samples_leaf=20, random_state=111)
    m.fit(X_flat_train, y_train)

    train_pred = m.predict(X_flat_train)
    test_pred  = m.predict(X_flat_test)
    train_mae  = mean_absolute_error(y_train, train_pred) * 100
    test_mae   = mean_absolute_error(y_test,  test_pred)  * 100
    dir_acc    = np.mean(np.sign(test_pred) == np.sign(y_test)) * 100
    # 과적합 지표: 훈련 MAE와 테스트 MAE의 차이가 클수록 과적합
    overfit    = test_mae - train_mae

    dt_search_results.append({
        'max_depth':   str(depth) if depth else '제한없음',
        '훈련 MAE(%)': round(train_mae, 3),
        '테스트 MAE(%)':round(test_mae,  3),
        '과적합 갭':   round(overfit,    3),
        '방향정확도':  round(dir_acc,    1),
    })
    print(f'depth={str(depth):<8}  '
          f'훈련={train_mae:.3f}%  테스트={test_mae:.3f}%  '
          f'갭={overfit:+.3f}%  방향={dir_acc:.1f}%')

dt_df = pd.DataFrame(dt_search_results)
print('\n=== 결정 트리 깊이별 성능 비교 ===')
print(dt_df.to_string(index=False))

best_depth = dt_df.loc[dt_df['테스트 MAE(%)'].idxmin(), 'max_depth']
print(f'\n★ 최적 max_depth: {best_depth}  → 최종 모델에 사용')

# ── 시각화 ──
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
x_labels = [r['max_depth'] for r in dt_search_results]
x_idx    = np.arange(len(x_labels))

axes[0].plot(x_idx, [r['훈련 MAE(%)'] for r in dt_search_results],
             marker='o', label='훈련 MAE', color='royalblue', linewidth=2)
axes[0].plot(x_idx, [r['테스트 MAE(%)'] for r in dt_search_results],
             marker='s', label='테스트 MAE', color='tomato', linewidth=2)
axes[0].set_xticks(x_idx)
axes[0].set_xticklabels(x_labels)
axes[0].set_title('깊이별 훈련 vs 테스트 MAE\n(간격이 클수록 과적합)')
axes[0].set_xlabel('max_depth')
axes[0].set_ylabel('MAE (%)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].bar(x_idx, [r['과적합 갭'] for r in dt_search_results],
            color=['salmon' if r['과적합 갭'] > 0.5 else 'seagreen'
                   for r in dt_search_results], alpha=0.85)
axes[1].set_xticks(x_idx)
axes[1].set_xticklabels(x_labels)
axes[1].set_title('깊이별 과적합 갭 (테스트 - 훈련 MAE)\n(낮을수록 일반화 잘 됨)')
axes[1].set_xlabel('max_depth')
axes[1].set_ylabel('과적합 갭 (%)')
axes[1].axhline(0, color='gray', linestyle='--')
axes[1].grid(True, alpha=0.3, axis='y')

plt.suptitle('결정 트리 max_depth 탐색', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### KNN 회귀 — 이웃 수(k) 탐색

| 파라미터 | 후보값 | 선택 기준 |
|----------|--------|-----------|
| n_neighbors (이웃 수 k) | 3 / 5 / 10 / **20** / 50 / 100 | 테스트 MAE 최소 |

> k가 작으면 → 가까운 소수 이웃만 참고 → 노이즈에 민감 (과적합)  
> k가 크면 → 너무 많은 이웃을 평균 → 특징이 뭉개짐 (과소적합)  
> **KNN의 핵심 원리**: "비슷한 상황(지표)의 과거 데이터들이 이후 비슷하게 움직인다"

In [ ]:
# ─────────────────────────────────────────
# [탐색] KNN 회귀 — 이웃 수(k) 비교
# ─────────────────────────────────────────
from sklearn.neighbors import KNeighborsRegressor

# ══════════════════════════════════════════
# KNN(K-Nearest Neighbors) 회귀란?
# ══════════════════════════════════════════
# "지금 상황과 가장 비슷했던 과거 k개 날을 찾아 그 결과의 평균을 예측"
#
# 예시 (k=5):
#   오늘 RSI=65, MACD=0.3, 거래량=1.5배 이런 상황에서...
#   과거에 비슷했던 5개 날을 검색 → 그 5일 후 등락률 평균 = 예측값
#
# 특징:
#   - '학습'이 없다. 데이터를 그대로 저장하고 예측 때마다 검색.
#   - 예측 속도가 느리지만 구현이 직관적.
#   - 데이터가 많을수록 좋아진다 (검색 후보가 많아지므로).

K_LIST = [3, 5, 10, 20, 50, 100]
knn_search_results = []

print('=== KNN 이웃 수(k) 탐색 ===\n')
for k in K_LIST:
    t0 = time.time()
    m = KNeighborsRegressor(n_neighbors=k, n_jobs=-1)
    m.fit(X_flat_train, y_train)
    pred    = m.predict(X_flat_test)
    mae     = mean_absolute_error(y_test, pred) * 100
    rmse    = np.sqrt(mean_squared_error(y_test, pred)) * 100
    dir_acc = np.mean(np.sign(pred) == np.sign(y_test)) * 100
    elapsed = time.time() - t0

    knn_search_results.append({
        'k': k, 'MAE(%)': round(mae, 3), 'RMSE(%)': round(rmse, 3),
        '방향정확도': round(dir_acc, 1), '예측시간(초)': round(elapsed, 1)
    })
    print(f'k={k:<4d}  MAE={mae:.3f}%  방향={dir_acc:.1f}%  ({elapsed:.1f}초)')

knn_df = pd.DataFrame(knn_search_results)
print('\n=== KNN k별 성능 비교 ===')
print(knn_df.to_string(index=False))
best_k = knn_df.loc[knn_df['MAE(%)'].idxmin(), 'k']
print(f'\n★ 최적 k: {best_k}  → 최종 모델에 사용')

# ── 시각화 ──
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ks = [r['k'] for r in knn_search_results]

axes[0].plot(ks, [r['MAE(%)'] for r in knn_search_results],
             marker='o', color='darkcyan', linewidth=2)
axes[0].axvline(best_k, color='red', linestyle='--', alpha=0.7, label=f'최적 k={best_k}')
axes[0].set_title('k vs MAE  (낮을수록 좋음)')
axes[0].set_xlabel('n_neighbors (k)')
axes[0].set_ylabel('MAE (%)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(ks, [r['방향정확도'] for r in knn_search_results],
             marker='s', color='darkcyan', linewidth=2)
axes[1].axhline(50, color='gray', linestyle='--', alpha=0.6, label='기준(50%)')
axes[1].axvline(best_k, color='red', linestyle='--', alpha=0.7)
axes[1].set_title('k vs 방향정확도  (높을수록 좋음)')
axes[1].set_xlabel('n_neighbors (k)')
axes[1].set_ylabel('방향정확도 (%)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('KNN 이웃 수(k) 탐색 결과', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 🌲 비교 모델: Random Forest & XGBoost

LSTM은 60일 시퀀스 전체를 입력으로 쓰지만, Random Forest / XGBoost는 **마지막 날의 기술적 지표만** 입력으로 사용합니다.

| 항목 | LSTM | Random Forest / XGBoost |
|------|------|--------------------------|
| 입력 | 60일 × 11개 특성 (시퀀스) | 당일 11개 기술적 지표 (플랫) |
| 시간 순서 반영 | ✅ | ❌ |
| 학습 속도 | 느림 (분 단위) | 빠름 (초 단위) |
| 해석 가능성 | 낮음 | 높음 (특성 중요도) |

In [ ]:
# ─────────────────────────────────────────
# 7단계: Random Forest & XGBoost 학습
# ─────────────────────────────────────────
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
import time

# ══════════════════════════════════════════
# RF / XGB 입력 방식 — LSTM과의 차이
# ══════════════════════════════════════════
# LSTM: (샘플수, 60일, 11개 지표) → 60일치 '흐름'을 통째로 입력
# RF/XGB: (샘플수, 11개 지표) → 오늘 하루의 지표만 입력 (플랫)
#
# X_train[:, -1, :] 의미:
#   - 첫 번째 : → 모든 샘플
#   - -1       → 60일 시퀀스 중 마지막 날(가장 최근 날)
#   - 두 번째 : → 11개 특성 전체
# 즉, '60일치 중 마지막 날의 지표'만 뽑아서 2D 배열로 만듦
X_flat_train = X_train[:, -1, :]
X_flat_test  = X_test[:, -1, :]
print(f'플랫 입력 형태: {X_flat_train.shape}  (샘플 x 11개 지표)\n')


# ══════════════════════════════════════════
# Random Forest 학습
# ══════════════════════════════════════════
# Random Forest란?
# '여러 전문가의 의견을 모아 다수결로 결론 내리는' 앙상블 방법.
#
# 동작 방식:
#   1. 학습 데이터를 랜덤 샘플링해 300개의 서로 다른 훈련 세트 생성 (배깅)
#   2. 각 훈련 세트로 결정 트리(Decision Tree) 1개씩 학습
#      → 트리마다 다른 데이터를 보므로 다양한 관점 형성
#   3. 예측 시: 300개 트리가 각자 등락률을 예측하고 평균 냄
#      → 한 트리가 틀려도 나머지가 보완 → 안정적
#
# 주요 하이퍼파라미터:
#   n_estimators=300 : 트리 300개 (탐색 결과 elbow point)
#   max_depth=8       : 트리 깊이 제한 → 너무 깊으면 훈련 데이터만 외움(과적합)
#   min_samples_leaf=20: 리프 노드(최종 예측 셀)에 최소 20개 샘플 필요
#                        → 너무 적은 샘플로 섣부른 예측 방지
#   n_jobs=-1         : CPU 코어 전부 사용 (병렬 학습)
print('Random Forest 학습 중...')
t0 = time.time()
rf = RandomForestRegressor(
    n_estimators=300, max_depth=8, min_samples_leaf=20,
    n_jobs=-1, random_state=111
)
rf.fit(X_flat_train, y_train)
print(f'  완료 ({time.time()-t0:.1f}초)\n')


# ══════════════════════════════════════════
# XGBoost 학습
# ══════════════════════════════════════════
# XGBoost(eXtreme Gradient Boosting)란?
# '이전 모델이 틀린 부분을 다음 모델이 집중적으로 보완하는' 부스팅 방법.
#
# RF와의 차이:
#   RF: 트리 300개를 독립적으로 만들고 평균 (병렬)
#   XGB: 트리를 순서대로 만들되, 앞 트리의 오차를 뒤 트리가 보정 (순차)
#        → '오답 노트를 보면서 공부하는 방식'
#
# 동작 방식:
#   1. 첫 번째 트리: 전체 데이터로 예측, 오차(잔차) 계산
#   2. 두 번째 트리: 첫 번째 트리의 '오차'를 타깃으로 학습
#   3. 세 번째 트리: 1+2번의 합산 오차를 타깃으로 학습
#   4. ... 500라운드까지 반복, 최종 예측 = 모든 트리 합산
#
# 주요 하이퍼파라미터:
#   n_estimators=500    : 최대 500라운드 (Early Stopping으로 약 350에서 실제 종료)
#   learning_rate=0.05  : 각 트리의 기여도를 5%로 제한
#                         → 작을수록 신중하게 학습, 클수록 빠르지만 불안정
#   max_depth=5         : 트리 깊이 (RF보다 얕게 → 개별 트리는 단순하게 유지)
#   subsample=0.8       : 각 라운드마다 훈련 데이터 80%만 사용 (과적합 방지)
#   colsample_bytree=0.8: 각 트리마다 특성 80%만 랜덤 사용 (다양성 확보)
#   early_stopping_rounds(탐색 시): 30라운드 개선 없으면 자동 종료
print('XGBoost 학습 중...')
t0 = time.time()
xgb = XGBRegressor(
    n_estimators=500, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    n_jobs=-1, random_state=111, verbosity=0
)
xgb.fit(X_flat_train, y_train,
        eval_set=[(X_flat_test, y_test)], verbose=False)
print(f'  완료 ({time.time()-t0:.1f}초)')


In [ ]:
# ─────────────────────────────────────────
# 7-2단계: 선형 회귀 계열 + 결정 트리 + KNN 학습
# ─────────────────────────────────────────
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
import time

# ══════════════════════════════════════════
# 6개 모델 한눈에 비교
# ══════════════════════════════════════════
# 모델            │ 핵심 아이디어                 │ 특징
# ─────────────────────────────────────────────────────────────────
# 선형 회귀       │ 입력 × 가중치 합              │ 가장 단순한 기준선 모델
# Ridge          │ 선형 회귀 + 계수 크기 패널티   │ 모든 특성 조금씩 활용
# Lasso          │ 선형 회귀 + 불필요 계수 제거   │ 중요 특성만 자동 선택
# ElasticNet     │ Ridge + Lasso 혼합            │ 두 방식 균형 있게 결합
# 결정 트리       │ 조건 분기 반복                │ 규칙 해석 가능, 과적합 주의
# KNN            │ 유사 과거 데이터 k개 평균      │ 학습 없음, 데이터가 곧 모델
# ─────────────────────────────────────────────────────────────────
# 공통점: 모두 '마지막 날의 기술적 지표 11개'만 입력 (LSTM과 달리 시계열 미반영)

print('=== 6개 모델 학습 시작 ===\n')

# ── 선형 회귀 ─────────────────────────────────────────
# 규제 없이 가중치를 최적화. 다른 모델의 성능 기준선(baseline) 역할.
print('선형 회귀 학습 중...', end='')
lr = LinearRegression()
lr.fit(X_flat_train, y_train)
print(' 완료')

# ── Ridge (규제 강도 α=0.1, 탐색에서 선택된 최적값) ──
# 가중치의 제곱합을 최소화 → 모든 계수가 조금씩 작아져 안정적
print('Ridge 학습 중...', end='')
ridge = Ridge(alpha=0.1, max_iter=2000)
ridge.fit(X_flat_train, y_train)
print(' 완료')

# ── Lasso (규제 강도 α=0.001, 탐색에서 선택된 최적값) ──
# 가중치의 절댓값 합을 최소화 → 중요하지 않은 계수가 정확히 0이 됨
# 어떤 기술적 지표가 '무의미'한지 자동으로 드러냄
print('Lasso 학습 중...', end='')
lasso = Lasso(alpha=0.001, max_iter=2000)
lasso.fit(X_flat_train, y_train)
print(' 완료')

# ── ElasticNet (l1_ratio=0.5: Ridge 50% + Lasso 50% 혼합) ──
print('ElasticNet 학습 중...', end='')
enet = ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=2000)
enet.fit(X_flat_train, y_train)
print(' 완료')

# ── 결정 트리 (max_depth=8, 탐색에서 선택된 최적값) ──
# 과적합 방지를 위해 깊이를 8로 제한, 리프 노드당 최소 20개 샘플
print('결정 트리 학습 중...', end='')
dt = DecisionTreeRegressor(max_depth=8, min_samples_leaf=20, random_state=111)
dt.fit(X_flat_train, y_train)
print(' 완료')

# ── KNN 회귀 (k=20, 탐색에서 선택된 최적값) ──
# 예측 기준일과 가장 유사한 과거 20일을 찾아 그 5일 후 등락률 평균
print('KNN 회귀 학습 중...', end='')
knn = KNeighborsRegressor(n_neighbors=20, n_jobs=-1)
knn.fit(X_flat_train, y_train)
print(' 완료')

print('\n=== Lasso가 0으로 만든 계수 (불필요하다고 판단된 특성) ===')
lasso_coef = pd.Series(lasso.coef_, index=FEATURES)
zeroed = lasso_coef[lasso_coef == 0]
active = lasso_coef[lasso_coef != 0]
print(f'활성 특성 ({len(active)}개): {list(active.index)}')
print(f'제거된 특성 ({len(zeroed)}개): {list(zeroed.index) if len(zeroed) else "없음"}')

print('\n=== Ridge 계수 크기 (큰 값 = 더 중요한 특성) ===')
ridge_coef = pd.Series(np.abs(ridge.coef_), index=FEATURES).sort_values(ascending=False)
for feat, val in ridge_coef.items():
    bar = chr(9608) * int(val * 500)
    print(f'  {feat:<15} {val:.5f}  {bar}')

In [ ]:
# ─────────────────────────────────────────
# 8단계: 9개 모델 최종 성능 비교
# ─────────────────────────────────────────
import matplotlib.gridspec as gridspec

# 9개 모델 예측값 계산
lstm_pred  = model.predict(X_test, verbose=0).flatten()
lr_pred    = lr.predict(X_flat_test)
ridge_pred = ridge.predict(X_flat_test)
lasso_pred = lasso.predict(X_flat_test)
enet_pred  = enet.predict(X_flat_test)
rf_pred    = rf.predict(X_flat_test)
xgb_pred   = xgb.predict(X_flat_test)
dt_pred    = dt.predict(X_flat_test)
knn_pred   = knn.predict(X_flat_test)

def evaluate(y_true, y_pred, name):
    return {
        '모델': name,
        'MAE(%)':       round(mean_absolute_error(y_true, y_pred) * 100, 2),
        'RMSE(%)':      round(np.sqrt(mean_squared_error(y_true, y_pred)) * 100, 2),
        '방향정확도(%)': round(np.mean(np.sign(y_pred) == np.sign(y_true)) * 100, 1),
    }

# 모델 카테고리별 색상 지정
MODEL_INFO = [
    # (이름,               예측값,     색상,          카테고리)
    ('LSTM',              lstm_pred,  'steelblue',   '딥러닝'),
    ('선형 회귀',          lr_pred,    '#e6a817',     '선형'),
    ('Ridge',             ridge_pred, '#d4850a',     '선형'),
    ('Lasso',             lasso_pred, '#b86b07',     '선형'),
    ('ElasticNet',        enet_pred,  '#9c5205',     '선형'),
    ('Random Forest',     rf_pred,    'seagreen',    '앙상블'),
    ('XGBoost',           xgb_pred,   'tomato',      '앙상블'),
    ('결정 트리',          dt_pred,    'mediumpurple','트리'),
    ('KNN',               knn_pred,   'darkcyan',    'KNN'),
]

results   = [evaluate(y_test, pred, name) for name, pred, color, cat in MODEL_INFO]
result_df = pd.DataFrame(results).set_index('모델')

print('=== 9개 모델 성능 비교 ===')
print(result_df.to_string())
print('\n* 방향정확도: 상승/하락 방향 예측 정확도 (50% 초과 = 랜덤보다 나음)')

# ── 전체 비교 막대 그래프 ──────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

names  = result_df.index.tolist()
maes   = result_df['MAE(%)'].tolist()
dirs   = result_df['방향정확도(%)'].tolist()
colors = [c for _, _, c, _ in MODEL_INFO]
x      = np.arange(len(names))

# MAE 비교 (낮을수록 좋음)
bars1 = axes[0].bar(x, maes, color=colors, alpha=0.85, edgecolor='white', linewidth=0.5)
axes[0].axhline(np.mean(maes), color='black', linestyle='--', linewidth=1, alpha=0.5,
                label=f'평균 MAE: {np.mean(maes):.2f}%')
axes[0].set_xticks(x)
axes[0].set_xticklabels(names, rotation=35, ha='right', fontsize=9)
axes[0].set_ylabel('MAE (%)')
axes[0].set_title('모델별 MAE 비교  (낮을수록 좋음)', fontsize=12)
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars1, maes):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                 f'{val:.2f}', ha='center', fontsize=8)

# 방향정확도 비교 (높을수록 좋음)
bars2 = axes[1].bar(x, dirs, color=colors, alpha=0.85, edgecolor='white', linewidth=0.5)
axes[1].axhline(50, color='gray', linestyle='--', linewidth=1.5, alpha=0.7, label='기준(50%)')
axes[1].set_xticks(x)
axes[1].set_xticklabels(names, rotation=35, ha='right', fontsize=9)
axes[1].set_ylabel('방향정확도 (%)')
axes[1].set_title('모델별 방향정확도  (50% 초과 = 유의미)', fontsize=12)
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3, axis='y')
axes[1].set_ylim(40, max(dirs) + 3)
for bar, val in zip(bars2, dirs):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                 f'{val:.1f}%', ha='center', fontsize=8)

plt.suptitle('9개 회귀 모델 종합 성능 비교', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# ── 카테고리별 평균 성능 요약 ──────────────────────────
print('\n=== 카테고리별 평균 성능 ===')
cat_df = pd.DataFrame(
    [(name, cat, ev['MAE(%)'], ev['방향정확도(%)'])
     for (name, _, _, cat), ev in zip(MODEL_INFO, results)],
    columns=['모델', '카테고리', 'MAE(%)', '방향정확도(%)']
).groupby('카테고리').agg({'MAE(%)': 'mean', '방향정확도(%)': 'mean'}).round(2)
print(cat_df.to_string())

# ── XGBoost / RF 특성 중요도 ──────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, model_obj, name, color in [
        (axes[0], xgb, 'XGBoost', 'tomato'),
        (axes[1], rf,  'Random Forest', 'seagreen')]:
    imp = pd.Series(model_obj.feature_importances_, index=FEATURES).sort_values()
    imp.plot(kind='barh', ax=ax, color=color, alpha=0.8)
    ax.set_title(f'{name} 특성 중요도')
    ax.set_xlabel('중요도')
    ax.grid(True, alpha=0.3, axis='x')

plt.suptitle('트리 계열 모델 특성 중요도 비교', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 📅 2025년 실전 검증

모델은 **2019~2024년 데이터**로 학습되었습니다.  
이 섹션에서는 학습에 사용되지 않은 **2025년 실제 주가**와 모델 예측치를 비교합니다.

| 항목 | 내용 |
|------|------|
| 입력 | 2025년 각 거래일 기준 직전 60거래일 지표 |
| 타겟 | 해당 시점 기준 5거래일 후 실제 등락률 |
| 검증 종목 | 학습에 사용된 20개 대형주 |


In [ ]:
# ─────────────────────────────────────────
# 9단계: 2025년 실전 검증 데이터 준비
# ─────────────────────────────────────────

def prepare_2025_validation(code, features, lookback, forecast, val_year=2025):
    """2025년 거래일에 해당하는 시퀀스와 실제 등락률을 반환."""
    df = load_stock(code)
    if df is None or len(df) < lookback + forecast + 50:
        return None, None, None

    df = add_all_indicators(df)
    avail = [f for f in features if f in df.columns]
    df = df[avail].dropna()
    if len(df) < lookback + forecast + 50:
        return None, None, None

    # 전체 기간(2019~2025) 데이터로 스케일러 맞춤 — 학습 때와 동일 방식
    scaler = MinMaxScaler()
    scaled = scaler.fit_transform(df)
    close_raw = df['Close'].values
    future_ret = (close_raw[forecast:] - close_raw[:-forecast]) / close_raw[:-forecast]

    X_val, y_val, dates_val = [], [], []
    for i in range(lookback, len(scaled) - forecast):
        # 시퀀스의 마지막 날(예측 기준일)이 2025년인 경우만 수집
        pred_date = df.index[i - 1]
        if pred_date.year == val_year:
            X_val.append(scaled[i - lookback:i])
            y_val.append(future_ret[i - lookback])
            dates_val.append(pred_date)

    if not X_val:
        return None, None, None
    return np.array(X_val), np.array(y_val), dates_val


# 20개 학습 종목에 대해 2025년 검증 데이터 수집
val_data = {}
for code in tqdm(TRAIN_STOCKS, desc='2025년 데이터 준비'):
    X_v, y_v, dates_v = prepare_2025_validation(code, FEATURES, LOOKBACK, FORECAST)
    if X_v is not None and len(X_v) > 0:
        val_data[code] = {'X': X_v, 'y': y_v, 'dates': dates_v}

print(f'\n2025년 데이터 확보: {len(val_data)}개 종목')
for code, d in list(val_data.items())[:3]:
    print(f'  {code}: {len(d["X"])}개 시퀀스  ({d["dates"][0].date()} ~ {d["dates"][-1].date()})')


In [ ]:
# ─────────────────────────────────────────
# 10단계: 2025년 예측 vs 실제 — 시계열 비교
# ─────────────────────────────────────────
stock_list_df = get_stock_list()
name_map_val  = stock_list_df.set_index('Code')['Name'].to_dict() if 'Name' in stock_list_df.columns else {}

# 시각화할 대표 종목 4개 선택
SHOW_CODES = ['005930', '000660', '005380', '035420']  # 삼성전자, SK하이닉스, 현대차, NAVER
SHOW_CODES = [c for c in SHOW_CODES if c in val_data]

# 각 모델별 스타일 정의 (가독성을 위해 굵기와 투명도 차별화)
VAL_MODELS = [
    # (이름,          예측함수,                                  색상,           선굵기, 투명도)
    ('실제 등락률',   None,                                      'black',        1.8,   1.0),
    ('LSTM',         lambda X: model.predict(X, verbose=0).flatten(), 'steelblue', 1.2, 0.9),
    ('선형 회귀',    lambda X: lr.predict(X[:, -1, :]),          '#e6a817',      1.0,   0.75),
    ('Ridge',        lambda X: ridge.predict(X[:, -1, :]),       '#d4850a',      1.0,   0.75),
    ('Lasso',        lambda X: lasso.predict(X[:, -1, :]),       '#b86b07',      1.0,   0.75),
    ('ElasticNet',   lambda X: enet.predict(X[:, -1, :]),        '#9c5205',      1.0,   0.75),
    ('Random Forest',lambda X: rf.predict(X[:, -1, :]),          'seagreen',     1.2,   0.8),
    ('XGBoost',      lambda X: xgb.predict(X[:, -1, :]),         'tomato',       1.2,   0.8),
    ('결정 트리',    lambda X: dt.predict(X[:, -1, :]),           'mediumpurple', 1.0,   0.75),
    ('KNN',          lambda X: knn.predict(X[:, -1, :]),          'darkcyan',     1.0,   0.75),
]

fig, axes = plt.subplots(len(SHOW_CODES), 1, figsize=(16, 5 * len(SHOW_CODES)), sharex=False)
if len(SHOW_CODES) == 1:
    axes = [axes]

for ax, code in zip(axes, SHOW_CODES):
    d      = val_data[code]
    dates  = d['dates']
    y_true = np.array(d['y'])

    # 실제 등락률 먼저 그리기
    ax.plot(dates, y_true * 100, color='black', lw=1.8, label='실제 등락률', zorder=5)
    ax.fill_between(dates, y_true*100, 0,
                    where=(y_true > 0), alpha=0.07, color='green')
    ax.fill_between(dates, y_true*100, 0,
                    where=(y_true < 0), alpha=0.07, color='red')

    # 각 모델 예측 그리기
    for name, pred_fn, color, lw, alpha in VAL_MODELS[1:]:
        pred = pred_fn(d['X'])
        ax.plot(dates, pred * 100, color=color, lw=lw, label=name, alpha=alpha)

    ax.axhline(0, color='gray', lw=0.8, linestyle='--')

    name_str = name_map_val.get(code, code)
    # LSTM 방향정확도를 대표로 표시
    lstm_p  = model.predict(d['X'], verbose=0).flatten()
    da_lstm = np.mean(np.sign(lstm_p) == np.sign(y_true)) * 100
    ax.set_title(f'{name_str}({code})  |  LSTM 방향정확도: {da_lstm:.1f}%', fontsize=11)
    ax.set_ylabel('5일 후 등락률 (%)')
    ax.legend(fontsize=7, loc='upper right', ncol=2)
    ax.grid(True, alpha=0.3)

plt.suptitle('2025년 실전 검증 — 9개 모델 예측 vs 실제 (5거래일 후 등락률)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────
# 11단계: 2025년 전 종목 종합 성능 + 월별 방향정확도
# ─────────────────────────────────────────

# 전체 2025년 데이터 합치기
all_X_val = np.vstack([d['X'] for d in val_data.values()])
all_y_val = np.concatenate([d['y'] for d in val_data.values()])
all_dates = [date for d in val_data.values() for date in d['dates']]

# 9개 모델 예측
preds_val = {
    'LSTM':          model.predict(all_X_val, verbose=0).flatten(),
    '선형 회귀':     lr.predict(all_X_val[:, -1, :]),
    'Ridge':         ridge.predict(all_X_val[:, -1, :]),
    'Lasso':         lasso.predict(all_X_val[:, -1, :]),
    'ElasticNet':    enet.predict(all_X_val[:, -1, :]),
    'Random Forest': rf.predict(all_X_val[:, -1, :]),
    'XGBoost':       xgb.predict(all_X_val[:, -1, :]),
    '결정 트리':     dt.predict(all_X_val[:, -1, :]),
    'KNN':           knn.predict(all_X_val[:, -1, :]),
}
colors_val = ['steelblue', '#e6a817', '#d4850a', '#b86b07', '#9c5205',
              'seagreen', 'tomato', 'mediumpurple', 'darkcyan']

print('=== 2025년 실전 검증 성능 (학습에 사용되지 않은 데이터) ===')
res_2025 = pd.DataFrame(
    [evaluate(all_y_val, pred, name) for name, pred in preds_val.items()]
).set_index('모델')
print(res_2025.to_string())
print('\n* 방향정확도 50% 초과 = 랜덤 추측보다 나음')

# ── 시각화 1: 학습셋 vs 2025년 MAE 비교 ──────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 학습셋 MAE (이미 위에서 계산한 result_df 재사용)
train_maes_dict = {
    'LSTM':          mean_absolute_error(y_test, lstm_pred)  * 100,
    '선형 회귀':     mean_absolute_error(y_test, lr_pred)    * 100,
    'Ridge':         mean_absolute_error(y_test, ridge_pred) * 100,
    'Lasso':         mean_absolute_error(y_test, lasso_pred) * 100,
    'ElasticNet':    mean_absolute_error(y_test, enet_pred)  * 100,
    'Random Forest': mean_absolute_error(y_test, rf_pred)    * 100,
    'XGBoost':       mean_absolute_error(y_test, xgb_pred)   * 100,
    '결정 트리':     mean_absolute_error(y_test, dt_pred)    * 100,
    'KNN':           mean_absolute_error(y_test, knn_pred)   * 100,
}
val_maes_dict = {name: mean_absolute_error(all_y_val, pred) * 100
                 for name, pred in preds_val.items()}

model_names = list(preds_val.keys())
x = np.arange(len(model_names))
w = 0.38

bars1 = axes[0].bar(x - w/2, [train_maes_dict[n] for n in model_names],
                    w, label='학습셋 (2019-24)', color=colors_val, alpha=0.6, edgecolor='white')
bars2 = axes[0].bar(x + w/2, [val_maes_dict[n]   for n in model_names],
                    w, label='실전 (2025)',       color=colors_val, alpha=0.95, edgecolor='white')
axes[0].set_xticks(x)
axes[0].set_xticklabels(model_names, rotation=35, ha='right', fontsize=8)
axes[0].set_ylabel('MAE (%)')
axes[0].set_title('학습셋 vs 실전(2025) MAE\n(막대 높이 차이 = 과적합 정도)', fontsize=11)
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3, axis='y')

# 2025년 방향정확도 비교
val_dirs = [np.mean(np.sign(pred) == np.sign(all_y_val)) * 100
            for pred in preds_val.values()]
bars3 = axes[1].bar(x, val_dirs, color=colors_val, alpha=0.85, edgecolor='white')
axes[1].axhline(50, color='gray', linestyle='--', linewidth=1.5, alpha=0.7, label='기준(50%)')
axes[1].set_xticks(x)
axes[1].set_xticklabels(model_names, rotation=35, ha='right', fontsize=8)
axes[1].set_ylabel('방향정확도 (%)')
axes[1].set_title('2025년 실전 방향정확도\n(50% 초과 = 무작위보다 나음)', fontsize=11)
axes[1].legend(fontsize=9)
axes[1].set_ylim(40, max(val_dirs) + 4)
axes[1].grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars3, val_dirs):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                 f'{val:.1f}%', ha='center', fontsize=8)

plt.suptitle('2025년 실전 검증 — 9개 모델 종합 결과', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# ── 시각화 2: 월별 방향정확도 히트맵 ─────────────────
months_df = pd.DataFrame({'date': pd.to_datetime(all_dates), 'actual': all_y_val})
for name, pred in preds_val.items():
    months_df[name] = pred
months_df['month'] = months_df['date'].dt.to_period('M')

# 월별로 각 모델의 방향정확도 계산
monthly_dir = months_df.groupby('month').apply(
    lambda g: pd.Series({
        name: np.mean(np.sign(g[name]) == np.sign(g['actual'])) * 100
        for name in preds_val.keys()
    })
, include_groups=False)

fig, ax = plt.subplots(figsize=(14, 5))
im = ax.imshow(monthly_dir.T.values, aspect='auto', cmap='RdYlGn', vmin=40, vmax=70)
ax.set_xticks(range(len(monthly_dir)))
ax.set_xticklabels([str(m) for m in monthly_dir.index], rotation=30, fontsize=9)
ax.set_yticks(range(len(preds_val)))
ax.set_yticklabels(list(preds_val.keys()), fontsize=9)
ax.set_title('2025년 월별 방향정확도 히트맵\n(초록 = 높음 / 빨강 = 낮음 / 기준선 50%)',
             fontsize=12, fontweight='bold')
plt.colorbar(im, ax=ax, label='방향정확도 (%)')

# 각 셀에 숫자 표시
for i, col in enumerate(monthly_dir.index):
    for j, name in enumerate(preds_val.keys()):
        val = monthly_dir.loc[col, name]
        ax.text(i, j, f'{val:.0f}', ha='center', va='center',
                fontsize=8, color='black' if 45 < val < 65 else 'white')

plt.tight_layout()
plt.show()

print('\n=== 최종 요약 ===')
best_model_test = res_2025['MAE(%)'].idxmin()
best_dir_test   = res_2025['방향정확도(%)'].idxmax()
print(f'2025년 실전 MAE 최저: {best_model_test} ({res_2025.loc[best_model_test, "MAE(%)"]:.2f}%)')
print(f'2025년 실전 방향정확도 최고: {best_dir_test} ({res_2025.loc[best_dir_test, "방향정확도(%)"]:.1f}%)')